**Manual pipeline run - step 7 of 7** (`7_Data_Quality_Report.ipynb`)

Run All. This is the last step. Nothing has to be typed in - every notebook in
this folder runs top to bottom on its own.

A copy of `codes\data quality\Compendium_Data_Quality.ipynb`, made by
`make_manual_pipeline_run.py`. Change that notebook, not this copy,
then re-run the script.

In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


In [ ]:
"""
CELL: Configuration - the same paths and constants notebook 1 uses, since this
notebook has to see exactly what notebook 1 would see.
"""
DATA_COLLECTOR_PATH = Path("C:/RAFFI/DSS-local") / "DATA COLLECTOR"
# Read the same way notebook 3 reads it, for the external-data structural
# check below - a folder per chapter, not every chapter has one.
EXTERNAL_DATA_PATH = DATA_COLLECTOR_PATH / "external data"
VALUE_CORRECTIONS_PATH = DATA_COLLECTOR_PATH / "value corrections.xlsx"
COMPENDIUM_PATH = Path("C:/RAFFI/DSS-local") / "COMPENDIUM ARAB SOCIETY - V2"
TRANSLATION_DICT_PATH = COMPENDIUM_PATH / "translation dict_V2.xlsx"
# Source citations already translated - kept out of the dictionary, read
# here so a citation notebook 1 would translate is not flagged.
SOURCE_TRANSLATIONS_PATH = COMPENDIUM_PATH / "source translations_V2.xlsx"

# Part 1 writes one report per chapter here: Logs\Population_Logs\.
LOGS_ROOT = COMPENDIUM_PATH / "Logs"


def chapter_logs_folder(chapter):
    """Population -> Logs/Population_Logs."""
    return LOGS_ROOT / f"{chapter}_Logs"

# Every questionnaire, Arabic or English, one subfolder per chapter - the same
# folder notebook 1 reads. A file's language is read off its own headers.
QUESTIONNAIRES_PATH = DATA_COLLECTOR_PATH / "datacollector_received_quest"
DEFAULT_LANGUAGE = "AR"

# Leave as None to check every chapter found, or restrict e.g. ["Population"].
CHAPTERS = None

# A fuzzy match must score at least this well (0 to 1) to be used automatically
# by the real pipeline. Below it is exactly what this notebook reviews - same
# number as notebook 1's, because a label just above the line needs no review
# and one just below it does.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Never fuzzy-matched by the real pipeline either: two citations differing by
# one digit score high enough to overwrite each other. A Source value still
# gets reviewed here if the dictionary has no *exact* translation for it, but
# never with a fuzzy-matched suggestion pre-filled.
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]



## Part 2 — the final files: completeness, contradictions, and the dashboard

Part 1 above reads the raw questionnaires, before any of the pipeline has
touched them. Part 2 reads the opposite end - the finished `<Chapter>_EN.xlsx`
files notebooks 1-4 produce - and answers a different question: not "is this
label right", but "is the finished data any good".

Three things, in one pass per chapter:

- **Completeness.** For every indicator x country, how many of the 17
  expected years (2010-2026) carry a value, banded `none` / `sparse` /
  `partial` / `strong` / `complete` rather than averaged into one number - a
  one-point series and a seventeen-point series must never be allowed to
  cancel out into something that reads as "half full". Which breakdowns a
  country actually supplies for an indicator, too.
- **Contradictions.** A reported total that does not match the sum of its own
  parts, a year-on-year change of more than 10x, a percentage indicator that
  also holds absolute counts, a percentage breakdown that does not add to
  100, a negative value or a percentage outside 0-100, and the same row
  reported twice with two different values.
- **The dashboard.** `docs/data.js` is regenerated from the same numbers, for
  `codes/docs/index.html`.

Nothing here suggests a correction the way Part 1 does - a spike or a
contradiction in the FINISHED data is a finding about the SOURCE, and fixing
it means going back to the questionnaire, not editing anything downstream.
`data_gaps_report.xlsx` and the dashboard are both read-only.


### Config

In [ ]:
"""
CELL: Configuration for the final-file checks - completeness, contradictions,
and the dashboard. COMPENDIUM_PATH is already set, above.
"""
import json

CODES_PATH = COMPENDIUM_PATH / "codes"
PROJECT_PATH = CODES_PATH / "data quality"
# The dashboard sits in codes/docs, not beside this notebook.
DOCS_PATH = CODES_PATH / "docs"
GAPS_REPORT_PATH = PROJECT_PATH / "data_gaps_report.xlsx"

# Every long file lives under here, one folder per chapter. Only the EN side is
# read below: every check here is written against English column names.
LONG_FILES_ROOT = COMPENDIUM_PATH / "Merged_long_files"

# The years the compendium expects a figure for. 2027-2030 exist as projection
# columns in the template and are not counted as gaps.
FIRST_YEAR, LAST_YEAR = 2010, 2026
EXPECTED_YEARS = list(range(FIRST_YEAR, LAST_YEAR + 1))

# Columns that are not a disaggregation.
FIXED = {"Indicator", "Country", "Chapter", "Year", "Value", "Source"}

# The value meaning "all of them", per dimension. A dimension absent from this
# map has no total by design - an indicator broken down that way (causes of
# death, economic activity, ...) has no meaningful aggregate row, so leaving
# it out is deliberate, not a gap in the map.
TOTAL_LABELS = {
    "Sex": "Both sexes",
    "Age Group": "Age Total",
    "Area": "Area Total",
    "Nationality": "Nationality Total",
    "Marital status": "Marital status Total",
    "Quintile": "Total",
    "Types of products/services": "Total",
}

# A year-on-year change of more than this is flagged. Deliberately loose: real
# populations do not change tenfold in a year, so a hit is a transcription
# error, a units change, or a break in the series worth a footnote.
SPIKE_FACTOR = 10

# How far a reported total may stray from the sum of its parts, as a fraction.
TOTAL_TOLERANCE = 0.01

# How far a set of percentages may stray from 100, in points.
PERCENT_TOLERANCE = 1.0

# The dashboard carries the worst N; the Excel report carries all of them.
MAX_DASHBOARD_ISSUES = 2000

# Series are grouped by how many of the 17 expected years they actually carry.
# Reporting one average percentage would let a one-point series and a full one
# cancel out into a number that reads as "half full"; these bands keep the
# difference visible. Ordered worst to best.
POINT_BANDS = [
    ("none", 0, 0),          # never reported at all
    ("sparse", 1, 6),        # too few points to read as a series
    ("partial", 7, 12),
    ("strong", 13, 16),
    ("complete", 17, 17),    # every expected year present
]


def band_of(points):
    for name, low, high in POINT_BANDS:
        if low <= points <= high:
            return name
    return "none"


### Helpers

In [ ]:
"""
CELL: Helpers for the final-file checks.

to_number() here is a lighter check than clean_one_value() above: this side
only ever needs a float or None for measurement across tens of thousands of
already-processed rows, never a suggested correction, so it does not carry
clean_one_value()'s unit-phrase, placeholder or sum-expression logic.
"""


def to_number(value):
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def headline_mask(table, dimensions):
    """True where a row is the headline figure: every dimension either empty or
    sitting at its own total. A dimension with no total label must be empty."""
    mask = pd.Series(True, index=table.index)
    for dimension in dimensions:
        column = table[dimension]
        total = TOTAL_LABELS.get(dimension)
        ok = column.isna()
        if total is not None:
            ok = ok | (column.astype(str).str.strip() == total)
        mask &= ok
    return mask


def missing_years_and_longest_gap(present):
    """Which expected years are absent, and the longest unbroken run of them."""
    missing = [y for y in EXPECTED_YEARS if y not in present]
    longest = run = 0
    for year in EXPECTED_YEARS:
        run = run + 1 if year not in present else 0
        longest = max(longest, run)
    return missing, longest


### Three more contradiction checks

In [ ]:
"""
CELL: Three more contradiction checks. Totals vs. parts, year-on-year jumps
and a percent indicator holding counts are not here - measure_chapter()
below already finds exactly those three, in one pass over the table, under
the "total mismatch", "spike" and "units" issue kinds.

Adapted to take the table `measure_chapter()` has already prepared (a
"number" column already parsed, EN column names only - column names were
being looked up in English there too, so nothing is lost) rather than
re-parsing Value from scratch a second time.
"""


def check_implausible_values(have, chapter):
    """Negative counts, and percentages outside 0-100.

    An indicator is treated as a percentage if its name says so - '%',
    'percentage', 'proportion'.
    """
    if "Indicator" not in have.columns:
        return []

    name = have["Indicator"].astype(str).str.lower()
    is_percentage = name.str.contains(r"%|percentage|proportion", regex=True, na=False)

    negative = have[have["number"] < 0]
    out_of_range = have[is_percentage & ((have["number"] < 0) | (have["number"] > 100))]

    issues = []
    for row in negative.itertuples(index=False):
        issues.append({
            "chapter": chapter, "kind": "implausible value",
            "indicator": getattr(row, "Indicator", ""), "country": getattr(row, "Country", ""),
            "detail": f"negative value: {row.number:,.1f}",
            "severity": round(abs(row.number), 1),
        })
    for row in out_of_range.itertuples(index=False):
        issues.append({
            "chapter": chapter, "kind": "implausible value",
            "indicator": getattr(row, "Indicator", ""), "country": getattr(row, "Country", ""),
            "detail": f"percentage outside 0-100: {row.number:,.1f}",
            "severity": round(max(-row.number, row.number - 100), 1),
        })
    return issues


def check_percentages_sum(have, chapter):
    """Percentage breakdowns that do not add up to 100 within their group.

    Only applied where a single dimension is clearly the thing being broken
    down, so a set of shares that silently drops a category is visible.
    """
    if "Indicator" not in have.columns:
        return []

    name = have["Indicator"].astype(str).str.lower()
    work = have[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    if work.empty:
        return []

    issues = []
    for dimension, total_label in TOTAL_LABELS.items():
        if dimension not in work.columns:
            continue
        part = work[~work[dimension].isin([total_label, "Age unknown"]) & work[dimension].notna()]
        if part.empty:
            continue
        keys = [c for c in ["Indicator", "Country", "Year", "Sex"]
                if c in part.columns and c != dimension]
        if not keys:
            continue
        sums = part.groupby(keys, dropna=False, observed=True)["number"].sum(min_count=1).dropna()
        bad = sums[(sums - 100).abs() > PERCENT_TOLERANCE]
        for key, adds_up_to in bad.items():
            record = dict(zip(keys, key if isinstance(key, tuple) else (key,)))
            issues.append({
                "chapter": chapter, "kind": "percentages do not sum to 100",
                "indicator": record.get("Indicator", ""), "country": record.get("Country", ""),
                "detail": f"{dimension} adds up to {adds_up_to:.1f}%",
                "severity": round(abs(adds_up_to - 100), 1),
            })
    return issues


def check_duplicate_rows(have, chapter):
    """The same dimensions and year appearing twice with DIFFERENT values.

    Identical duplicates are harmless noise; conflicting ones mean one of the
    two figures is wrong and there is no way to tell which downstream.
    """
    keys = [c for c in have.columns if c not in FIXED and c != "number"]
    if not keys:
        return []

    counts = have.groupby(keys, dropna=False, observed=True)["number"].nunique()
    conflicting = counts[counts > 1]
    if conflicting.empty:
        return []

    issues = []
    for key, distinct_values in conflicting.head(500).items():
        record = dict(zip(keys, key if isinstance(key, tuple) else (key,)))
        issues.append({
            "chapter": chapter, "kind": "conflicting duplicate rows",
            "indicator": record.get("Indicator", ""), "country": record.get("Country", ""),
            "detail": (f"{distinct_values} distinct values reported for the same row - "
                       f"{', '.join(f'{k}={v}' for k, v in record.items() if k not in ('Indicator', 'Country'))}"),
            "severity": int(distinct_values),
        })
    return issues


### measure_chapter()

In [ ]:
"""
CELL: measure_chapter() - every metric for one chapter's long file.
"""


def measure_chapter(path):
    """Returns (series, dimensions, issues) for one <Chapter>_EN.xlsx."""
    chapter = path.name[: -len("_EN.xlsx")]
    table = pd.read_excel(path, engine="openpyxl")
    if "Indicator" not in table.columns:
        logger.warning(f"{path.name}: no Indicator column, skipping")
        return [], [], []

    dimensions = [c for c in table.columns if c not in FIXED]
    table["number"] = table["Value"].map(to_number)
    have = table[table["number"].notna()].copy()
    have["Year"] = have["Year"].astype(int)
    have["is_headline"] = headline_mask(have, dimensions)

    logger.info(f"{chapter}: {len(table):,} rows, {len(have):,} with a value, "
                f"{len(dimensions)} dimension(s)")

    series, dimension_rows, issues = [], [], []

    # -------------------------------------------------- levels 1 and 2
    # Every indicator x country the chapter could report, not only the pairs
    # that reported something. A pair with nothing is a real finding - a
    # 0-point series - and leaving it out of the denominator would quietly
    # flatter every summary built on top.
    reported = {key: group for key, group
                in have.groupby(["Indicator", "Country"], observed=True)}
    all_indicators = sorted(table["Indicator"].dropna().unique())
    all_countries = sorted(table["Country"].dropna().unique())

    for indicator in all_indicators:
        for country in all_countries:
            group = reported.get((indicator, country))

            if group is None:
                years_any, years_head = set(), set()
            else:
                years_any = set(group["Year"]) & set(EXPECTED_YEARS)
                years_head = set(group.loc[group["is_headline"], "Year"]) & set(EXPECTED_YEARS)
            missing, longest = missing_years_and_longest_gap(years_any)

            series.append({
                "chapter": chapter, "indicator": indicator, "country": country,
                "points": len(years_any),
                "headline_points": len(years_head),
                "band": band_of(len(years_any)),
                "coverage": round(len(years_any) / len(EXPECTED_YEARS) * 100, 1),
                "first_year": min(years_any) if years_any else None,
                "last_year": max(years_any) if years_any else None,
                "longest_gap": longest, "missing_years": missing,
                "values": 0 if group is None else len(group),
            })

            if group is None:
                continue
            for dimension in dimensions:
                total = TOTAL_LABELS.get(dimension)
                values = group[dimension].dropna().astype(str).str.strip()
                if total is not None:
                    values = values[values != total]
                if values.empty:
                    continue
                dimension_rows.append({
                    "chapter": chapter, "indicator": indicator, "country": country,
                    "dimension": dimension, "categories": values.nunique(),
                })

    # ------------------------------------------------------- dubious spikes
    # One value per series per year first: the same dimensions and year can
    # appear twice in these files, and without this the comparison below would
    # pit two rows from the SAME year against each other.
    keys = ["Indicator", "Country"] + dimensions
    tidy = (have.groupby(keys + ["Year"], dropna=False, observed=True)["number"]
            .first().reset_index().sort_values(keys + ["Year"]))
    grouped = tidy.groupby(keys, dropna=False, observed=True)
    tidy["previous"] = grouped["number"].shift(1)
    tidy["previous_year"] = grouped["Year"].shift(1)
    candidates = tidy[tidy["previous"].notna() & (tidy["previous"] != 0)
                      & (tidy["Year"] > tidy["previous_year"])].copy()
    ratio = candidates["number"] / candidates["previous"]
    for _, row in candidates[(ratio > SPIKE_FACTOR) | (ratio < 1 / SPIKE_FACTOR)].iterrows():
        issues.append({
            "chapter": chapter, "kind": "spike",
            "indicator": row["Indicator"], "country": row["Country"],
            "detail": (f"{int(row['previous_year'])}: {row['previous']:,.0f}"
                       f"  ->  {int(row['Year'])}: {row['number']:,.0f}"),
            "severity": round(abs(row["number"] / row["previous"]), 1),
        })

    # ------------------------------- percentage indicators holding counts
    name = have["Indicator"].astype(str).str.lower()
    percentages = have[name.str.contains(r"%|percentage|proportion", regex=True, na=False)]
    for indicator, group in percentages.groupby("Indicator", observed=True):
        over = group["number"] > 100
        if over.any() and not over.all():
            issues.append({
                "chapter": chapter, "kind": "units",
                "indicator": indicator, "country": "(several)",
                "detail": (f"{int(over.sum()):,} of {len(group):,} rows exceed 100 "
                           f"(largest {group['number'].max():,.0f}) - percentages and "
                           f"counts mixed in one indicator"),
                "severity": round(float(group["number"].max()), 0),
            })

    # ------------------------------ totals that contradict their own parts
    for dimension, total in TOTAL_LABELS.items():
        if dimension not in have.columns:
            continue
        keys2 = [c for c in ["Indicator", "Country", "Year", "Sex"]
                 if c in have.columns and c != dimension]
        work = have[["number", dimension] + keys2]
        label = work[dimension].astype(str).str.strip()
        totals = work[label == total].groupby(keys2, observed=True)["number"].first()
        parts = (work[work[dimension].notna() & (label != total) & (label != "Age unknown")]
                 .groupby(keys2, observed=True)["number"].sum(min_count=1))
        shared = totals.index.intersection(parts.index)
        if not len(shared):
            continue
        compare = pd.DataFrame({"total": totals.loc[shared], "parts": parts.loc[shared]})
        compare = compare[compare["total"].abs() > 0]
        gap = (compare["parts"] - compare["total"]).abs() / compare["total"].abs()
        for key, off in gap[gap > TOTAL_TOLERANCE].items():
            record = dict(zip(keys2, key if isinstance(key, tuple) else (key,)))
            issues.append({
                "chapter": chapter, "kind": "total mismatch",
                "indicator": record.get("Indicator", ""),
                "country": record.get("Country", ""),
                "detail": (f"{dimension} {record.get('Year', '')}: reported "
                           f"{compare.loc[key, 'total']:,.0f} vs parts summing to "
                           f"{compare.loc[key, 'parts']:,.0f}"),
                "severity": round(float(off) * 100, 1),
            })

    # ---------------------------------- the three further contradiction checks
    issues += check_implausible_values(have, chapter)
    issues += check_percentages_sum(have, chapter)
    issues += check_duplicate_rows(have, chapter)

    return series, dimension_rows, issues


### Run - Part 2

In [ ]:
"""
CELL: Measure every chapter's final file.
"""
files = sorted(LONG_FILES_ROOT.glob("*_Merged_long_files/*_EN.xlsx"))
print(f"Reading {len(files)} chapter file(s) from {LONG_FILES_ROOT}\n")

all_series, all_dimensions, all_issues = [], [], []
for path in files:
    s, d, i = measure_chapter(path)
    all_series += s
    all_dimensions += d
    all_issues += i

SERIES = pd.DataFrame(all_series)
DIMENSIONS = pd.DataFrame(all_dimensions)
ISSUES = pd.DataFrame(all_issues)

if SERIES.empty:
    print("\nNo data. Run the pipeline notebooks first.")
else:
    print(f"\n{len(SERIES):,} series - {len(DIMENSIONS):,} disaggregation entries "
          f"- {len(ISSUES):,} flagged figures")

    print(f"\nHow many of the {len(EXPECTED_YEARS)} years each series actually has:")
    counts = SERIES["points"].value_counts().sort_index(ascending=False)
    for points, n in counts.items():
        bar = "#" * max(1, round(n / counts.max() * 40))
        note = "  <- complete" if points == len(EXPECTED_YEARS) else (
               "  <- nothing at all" if points == 0 else "")
        print(f"   {points:>2} points  {n:>5,}  {bar}{note}")

    print("\nBy band:")
    for name, low, high in POINT_BANDS:
        n = (SERIES["band"] == name).sum()
        span = f"{low}" if low == high else f"{low}-{high}"
        print(f"   {name:<9} ({span:>5} points)  {n:>5,}  {n/len(SERIES):>5.1%}")

    print("\nComplete series by chapter:")
    for chapter, group in SERIES.groupby("chapter"):
        full = (group["points"] == len(EXPECTED_YEARS)).sum()
        none = (group["points"] == 0).sum()
        print(f"   {chapter:<12} {full:>4,} complete - {none:>4,} empty - "
              f"{len(group):>4,} total")

    print(f"\nSeries reporting nothing since 2019: "
          f"{(SERIES['last_year'] < 2020).sum():,}")
    if len(ISSUES):
        print("\nFlagged figures by kind:")
        print(ISSUES["kind"].value_counts().to_string())


### The Excel report

In [ ]:
"""
CELL: Write the Excel report.
"""
if not SERIES.empty:
    with pd.ExcelWriter(GAPS_REPORT_PATH, engine="openpyxl") as writer:
        summary = pd.DataFrame(
            [{"metric": "series (every indicator x country)", "value": len(SERIES)}]
            + [{"metric": f"{name} ({low} points)" if low == high
                          else f"{name} ({low}-{high} points)",
                "value": int((SERIES["band"] == name).sum())}
               for name, low, high in reversed(POINT_BANDS)]
            + [{"metric": "series stale since 2019",
                "value": int((SERIES["last_year"] < 2020).sum())},
               {"metric": "flagged figures", "value": len(ISSUES)}])
        summary.to_excel(writer, sheet_name="summary", index=False)

        # One row per point count, so the shape of the distribution is readable
        # without opening the dashboard.
        distribution = (SERIES["points"].value_counts().reindex(
            range(len(EXPECTED_YEARS), -1, -1), fill_value=0)
            .rename_axis("points").reset_index(name="series"))
        distribution["share"] = (distribution["series"] / len(SERIES)).round(3)
        distribution.to_excel(writer, sheet_name="points distribution", index=False)

        (SERIES.pivot_table(index="country", columns="band", values="indicator",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by country"))
        (SERIES.pivot_table(index="indicator", columns="band", values="country",
                            aggfunc="count", fill_value=0)
         .reindex(columns=[b[0] for b in reversed(POINT_BANDS)], fill_value=0)
         .to_excel(writer, sheet_name="bands by indicator"))

        out = SERIES.copy()
        out["missing_years"] = out["missing_years"].map(
            lambda years: ", ".join(str(y) for y in years))
        out.sort_values("points").to_excel(writer, sheet_name="series", index=False)

        if not DIMENSIONS.empty:
            DIMENSIONS.to_excel(writer, sheet_name="disaggregation", index=False)
        if not ISSUES.empty:
            (ISSUES.sort_values("severity", ascending=False)
             .to_excel(writer, sheet_name="flagged figures", index=False))

    print(f"Wrote {GAPS_REPORT_PATH.name}")
    print("  sheets: summary, points distribution, series, bands by country,")
    print("          bands by indicator, disaggregation, flagged figures")


### The dashboard

In [ ]:
"""
CELL: Regenerate the dashboard's data file.
"""


def write_dashboard_data():
    """Write docs/data.js as index-referenced arrays.

    Repeating "chapter"/"indicator"/"country" as JSON keys across hundreds of
    series and thousands of issues costs about 1.5 MB; the same content as
    arrays with a lookup table is a fraction of that, and the page rehydrates
    it on load.
    """
    def index_of(values):
        ordered = sorted(set(values))
        return ordered, {v: i for i, v in enumerate(ordered)}

    chapters, chapter_ix = index_of(SERIES["chapter"])
    countries, country_ix = index_of(SERIES["country"])
    indicators, indicator_ix = index_of(SERIES["indicator"])
    dimensions, dimension_ix = (index_of(DIMENSIONS["dimension"])
                                if not DIMENSIONS.empty else ([], {}))
    kinds, kind_ix = index_of(ISSUES["kind"]) if not ISSUES.empty else ([], {})

    series_rows = [[
        chapter_ix[r.chapter], indicator_ix[r.indicator], country_ix[r.country],
        int(r.points), int(r.headline_points),
        int(r.first_year) if pd.notna(r.first_year) else None,
        int(r.last_year) if pd.notna(r.last_year) else None,
        int(r.longest_gap), int(r.values), [int(y) for y in (r.missing_years or [])],
    ] for r in SERIES.itertuples(index=False)]

    dimension_rows = [[
        chapter_ix[r.chapter], indicator_ix[r.indicator], country_ix[r.country],
        dimension_ix[r.dimension], int(r.categories),
    ] for r in DIMENSIONS.itertuples(index=False)] if not DIMENSIONS.empty else []

    worst = (ISSUES.sort_values("severity", ascending=False).head(MAX_DASHBOARD_ISSUES)
             if not ISSUES.empty else ISSUES)
    issue_rows = [[
        chapter_ix.get(r.chapter, 0), kind_ix[r.kind],
        indicator_ix.get(r.indicator, -1), country_ix.get(r.country, -1),
        float(r.severity), str(r.detail),
    ] for r in worst.itertuples(index=False)] if not worst.empty else []

    payload = {
        "generated": pd.Timestamp.now().strftime("%d %B %Y"),
        "years": EXPECTED_YEARS,
        "chapters": chapters, "countries": countries,
        "indicators": indicators, "dimensions": dimensions, "kinds": kinds,
        "series": series_rows, "dims": dimension_rows, "issues": issue_rows,
        "issueCounts": ({k: int(v) for k, v in ISSUES["kind"].value_counts().items()}
                        if not ISSUES.empty else {}),
        "issuesShown": len(issue_rows), "issuesTotal": int(len(ISSUES)),
        "bands": [[name, low, high] for name, low, high in POINT_BANDS],
    }

    DOCS_PATH.mkdir(parents=True, exist_ok=True)
    path = DOCS_PATH / "data.js"
    path.write_text("window.GAPS = " + json.dumps(payload, separators=(",", ":")) + ";\n",
                    encoding="utf-8")
    return path


if not SERIES.empty:
    written = write_dashboard_data()
    print(f"Wrote {written.relative_to(CODES_PATH)} "
          f"({written.stat().st_size / 1024:,.0f} KB)")
    print("\nOpen docs/index.html to view the dashboard.")
